In [0]:
%sql

CREATE OR REPLACE TEMP VIEW stg_pos_order AS SELECT
    pos_order_id,
    pos_order_name,
    pos_order_state,
    pos_reference,
    move_id,
    move_name,
    CAST(CASE 
        WHEN pos_order_name LIKE '%REFUND%' THEN TRUE
        ELSE FALSE
        END AS BOOLEAN
        ) AS is_refunded,
    CAST(CASE 
        WHEN pos_order_name LIKE '%RETURN%' THEN regexp_replace(pos_order_name, ' REFUND', '')
        ELSE ''
        END AS string
    ) AS origin,
    created_at,
    updated_at,
    is_updated,
    is_deleted
FROM bakery_pos_develop.silver.pos_order
QUALIFY ROW_NUMBER() OVER(
    PARTITION BY pos_order_id
    ORDER BY updated_at DESC
) = 1

In [0]:
%sql

CREATE TABLE IF NOT EXISTS bakery_pos_develop.gold.dim_pos_order (
    pos_order_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    pos_order_id BIGINT,
    pos_order_name STRING,
    pos_order_state STRING,
    pos_reference STRING,
    move_id BIGINT,
    move_name STRING,
    is_refunded BOOLEAN,
    origin STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    is_updated BOOLEAN,
    is_deleted BOOLEAN,
    PRIMARY KEY (pos_order_sk)
)

In [0]:
%sql
MERGE INTO bakery_pos_develop.gold.dim_pos_order AS tgt
USING stg_pos_order AS src
ON tgt.pos_order_id = src.pos_order_id

WHEN MATCHED AND src.updated_at > tgt.updated_at THEN
    UPDATE SET
        tgt.pos_order_name = src.pos_order_name,
        tgt.pos_order_state = src.pos_order_state,
        tgt.pos_reference = src.pos_reference,
        tgt.move_id = src.move_id,
        tgt.move_name = src.move_name,
        tgt.is_refunded = src.is_refunded,
        tgt.origin = src.origin,
        tgt.created_at = src.created_at,
        tgt.updated_at = src.updated_at,
        tgt.is_updated = src.is_updated,
        tgt.is_deleted = src.is_deleted

WHEN NOT MATCHED THEN
    INSERT (
        pos_order_id,
        pos_order_name,
        pos_order_state,
        pos_reference,
        move_id,
        move_name,
        is_refunded,
        origin,
        created_at,
        updated_at,
        is_updated,
        is_deleted
    )
    VALUES (
        src.pos_order_id,
        src.pos_order_name,
        src.pos_order_state,
        src.pos_reference,
        src.move_id,
        src.move_name,
        src.is_refunded,
        src.origin,
        src.created_at,
        src.updated_at,
        src.is_updated,
        src.is_deleted
    )